# s15 — Download BLP neuron meshes

Downloads the meshes of the right-lateralized bilateral (BLP_R) neurons listed in
`Processed_Data/BLP_R_root_ids.csv` (produced by
`Figures/fig_6B_postPI_prePI_BLP_neurons.m`), saving them under
`Processed_Data/fig6c_neuron_mesh_BLP` (used by `fig_6C_BLP_neurons.m`).

In [ ]:
from fafbseg import flywire
import navis
import numpy as np
import os

In [ ]:
# Base folder holding the BLP root_id list and the output meshes.
# baseDir is resolved from the notebook's location (run from Data_Processing/),
# so the code runs after download without editing any path.
baseDir = os.path.dirname(os.getcwd())
processed_dir = os.path.join(baseDir, "Processed_Data")


def download_meshes(root_id_csv, save_folder):
    """Download the mesh of every root_id listed in root_id_csv into save_folder.
    Ids whose vertices/faces files already exist are skipped."""
    os.makedirs(save_folder, exist_ok=True)

    # delimiter ',' : CSV is comma-separated; read the ids as int64
    data = np.loadtxt(root_id_csv, delimiter=',', dtype=np.int64)
    data = np.atleast_1d(data)  # handle a single-row file

    total_count = len(data)
    print(f"Starting to process {total_count} neurons...")

    for i in range(total_count):
        root_id = data[i]

        # Build the output file paths in advance
        v_path = os.path.join(save_folder, f"{root_id}_vertices.csv")
        f_path = os.path.join(save_folder, f"{root_id}_faces.csv")

        # Skip safely only if both vertices and faces already exist
        if os.path.exists(v_path) and os.path.exists(f_path):
            print(f"[{i+1}/{total_count}] Skip: {root_id} (already exists)")
            continue

        # Download and save only when the files are missing
        try:
            neuron = flywire.get_mesh_neuron(root_id)

            np.savetxt(v_path, neuron.vertices, delimiter=",")
            np.savetxt(f_path, neuron.faces, delimiter=",", fmt='%d')  # faces as integers

            print(f"[{i+1}/{total_count}] Saved: {root_id}")

        except Exception as e:
            print(f"[{i+1}/{total_count}] Error processing {root_id}: {e}")

In [ ]:
# Right-lateralized bilateral (BLP_R) neurons
print('BLP_R')
download_meshes(
    os.path.join(processed_dir, 'BLP_R_root_ids.csv'),
    os.path.join(processed_dir, 'fig6c_neuron_mesh_BLP'),
)